In [2]:
import torch

# GPU 사용 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("사용 장치:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능: True
사용 장치: cuda
GPU: NVIDIA GeForce RTX 4060


In [ ]:
# =============================================================================
# 05_pytorch_model_training.ipynb
# PyTorch 감정인식 모델 본학습
#
# - 기존 전처리된 grayscale PNG 사용
# - 기존 AI-Hub JSON 라벨 사용
# - 새로운 이미지 전처리 없음
# - 새로운 데이터 폴더 생성 없음
# - MobileNetV3 Small / EfficientNetB0 / ResNet18
# - pretrained weight 사용 안 함
# - 이전 1 Epoch 스크리닝 가중치 사용 안 함
# =============================================================================


# =============================================================================
# 필요한 라이브러리
# =============================================================================

from pathlib import Path
import json
import time
import gc

import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models



# =============================================================================
# 1. 학습 환경 및 기존 데이터 연결
# =============================================================================

display(Markdown("""
# 1. 학습 환경 및 기존 데이터 연결

이미 전처리가 완료된 **224×224 grayscale PNG**와  
기존 AI-Hub JSON의 감정 라벨을 연결한다.

- 이미지 resize를 다시 하지 않는다.
- grayscale 변환을 다시 하지 않는다.
- 기존 이미지 파일을 복사하거나 이동하지 않는다.
- `filename`을 기준으로 이미지와 라벨을 연결한다.
- 감정 정답은 기존 AI-Hub의 `faceExp_uploader`를 사용한다.
- Training 전체 223,578장과 Validation 전체 52,126장을 사용한다.
"""))


# -----------------------------------------------------------------------------
# 프로젝트 경로
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"D:\emotion_recognition_project"
)


# 이미 만들어 둔 grayscale 이미지
TRAIN_IMAGE_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "processed"
    / "grayscale_png"
    / "train"
)

VALID_IMAGE_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "processed"
    / "grayscale_png"
    / "valid"
)


# 기존 AI-Hub 라벨
TRAIN_LABEL_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "labels"
    / "train"
)

VALID_LABEL_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "labels"
    / "valid"
)


# 기존 프로젝트 결과 저장 폴더
MODEL_DIR = (
    PROJECT_ROOT
    / "05_models"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "06_outputs"
)


# 폴더가 이미 존재하면 그대로 사용
# 새로운 데이터 폴더를 만드는 코드가 아님
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# -----------------------------------------------------------------------------
# GPU 설정
# -----------------------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("사용 장치:", device)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# 입력 이미지 크기가 항상 224×224이므로
# cuDNN이 효율적인 연산 방법을 선택하도록 설정
torch.backends.cudnn.benchmark = True



# -----------------------------------------------------------------------------
# 감정 클래스
# -----------------------------------------------------------------------------

CLASS_NAMES = [
    "기쁨",
    "당황",
    "분노",
    "불안",
    "상처",
    "슬픔",
    "중립"
]


# 문자열 감정을 숫자로 변경
# CrossEntropyLoss는 숫자 정답을 사용하기 때문
LABEL_TO_INDEX = {

    label: index

    for index, label
    in enumerate(CLASS_NAMES)
}


NUM_CLASSES = len(
    CLASS_NAMES
)



# -----------------------------------------------------------------------------
# 본학습 공통 조건
# -----------------------------------------------------------------------------

BATCH_SIZE = 32

EPOCHS = 30

LEARNING_RATE = 0.001

PATIENCE = 5



# -----------------------------------------------------------------------------
# JSON 내부에서 filename + faceExp_uploader 찾기
#
# JSON의 최상위 구조가 list이든 dict이든
# 내부를 재귀적으로 탐색한다.
# -----------------------------------------------------------------------------

def collect_label_records(
    obj,
    rows
):

    # 현재 객체가 dictionary인 경우
    if isinstance(obj, dict):


        # 우리가 필요한 두 정보가 모두 있는지 확인
        if (
            "filename" in obj
            and
            "faceExp_uploader" in obj
        ):

            filename = obj.get(
                "filename"
            )

            label = obj.get(
                "faceExp_uploader"
            )


            # 정상적인 파일명이고
            # 우리가 사용하는 7개 감정 중 하나일 때만 저장
            if (
                isinstance(filename, str)
                and
                label in LABEL_TO_INDEX
            ):

                rows.append({

                    "filename":
                        filename,

                    "label":
                        label,

                    "label_index":
                        LABEL_TO_INDEX[label]
                })


        # dictionary 내부에 또 다른
        # dictionary/list가 있을 수 있으므로 계속 탐색
        for value in obj.values():

            collect_label_records(
                value,
                rows
            )


    # 현재 객체가 list인 경우
    elif isinstance(obj, list):

        for item in obj:

            collect_label_records(
                item,
                rows
            )



# -----------------------------------------------------------------------------
# 라벨 폴더 전체 JSON → DataFrame
# -----------------------------------------------------------------------------

def make_label_dataframe(
    label_dir
):

    rows = []


    # 기존 라벨 폴더 내부 JSON을 읽는다.
    for json_path in Path(
        label_dir
    ).rglob("*.json"):


        with open(
            json_path,
            "r",
            encoding="utf-8-sig"
        ) as file:

            data = json.load(
                file
            )


        collect_label_records(
            data,
            rows
        )


    label_df = pd.DataFrame(
        rows
    )


    # 동일 filename이 여러 번 수집되는 것을 방지
    if not label_df.empty:

        label_df = (
            label_df
            .drop_duplicates(
                subset="filename",
                keep="first"
            )
            .reset_index(drop=True)
        )


    return label_df



# -----------------------------------------------------------------------------
# 기존 grayscale PNG와 라벨 연결
# -----------------------------------------------------------------------------

def connect_images_and_labels(
    image_dir,
    label_df
):

    # 이미 만들어져 있는 PNG 경로만 읽는다.
    # 파일을 수정하거나 이동하지 않는다.
    image_paths = list(
        Path(image_dir).rglob("*.png")
    )


    # PNG 파일을 DataFrame으로 구성
    image_df = pd.DataFrame({

        "image_path": [
            str(path)
            for path in image_paths
        ],

        # 원본 jpg와 현재 png는
        # 확장자만 다르므로 stem을 사용
        "stem": [
            path.stem
            for path in image_paths
        ]
    })


    if label_df.empty:

        return pd.DataFrame()


    label_df = label_df.copy()


    # 예:
    # abc.jpg → abc
    # abc.png → abc
    #
    # 확장자를 제외한 파일명으로 연결
    label_df["stem"] = (
        label_df["filename"]
        .map(
            lambda x:
            Path(x).stem
        )
    )


    connected_df = image_df.merge(

        label_df[
            [
                "stem",
                "label",
                "label_index"
            ]
        ],

        on="stem",

        how="inner"
    )


    # 동일 이미지가 혹시 중복 연결되는 것을 방지
    connected_df = (
        connected_df
        .drop_duplicates(
            subset="image_path",
            keep="first"
        )
        .reset_index(drop=True)
    )


    return connected_df



# -----------------------------------------------------------------------------
# 실제 이미지 + 라벨 연결
# -----------------------------------------------------------------------------

print(
    "\nTraining 라벨 읽는 중..."
)


train_label_df = (
    make_label_dataframe(
        TRAIN_LABEL_DIR
    )
)


print(
    "Training 라벨:",
    f"{len(train_label_df):,}"
)


print(
    "\nValidation 라벨 읽는 중..."
)


valid_label_df = (
    make_label_dataframe(
        VALID_LABEL_DIR
    )
)


print(
    "Validation 라벨:",
    f"{len(valid_label_df):,}"
)



print(
    "\nTraining grayscale PNG와 라벨 연결 중..."
)


train_df = (
    connect_images_and_labels(
        TRAIN_IMAGE_DIR,
        train_label_df
    )
)


print(
    "Training 연결:",
    f"{len(train_df):,}"
)



print(
    "\nValidation grayscale PNG와 라벨 연결 중..."
)


valid_df = (
    connect_images_and_labels(
        VALID_IMAGE_DIR,
        valid_label_df
    )
)


print(
    "Validation 연결:",
    f"{len(valid_df):,}"
)



# -----------------------------------------------------------------------------
# 본학습에 들어가기 전 최소 확인
#
# 전처리를 다시 검증하는 과정이 아니라
# 기존에 확정한 데이터 개수와 연결 결과가 같은지만 확인한다.
# -----------------------------------------------------------------------------

EXPECTED_TRAIN = 223_578

EXPECTED_VALID = 52_126


if len(train_df) != EXPECTED_TRAIN:

    raise RuntimeError(

        "\nTraining 데이터 연결 개수가 맞지 않습니다.\n"

        f"예상: {EXPECTED_TRAIN:,}\n"

        f"현재: {len(train_df):,}\n"

        "학습을 시작하지 않고 중단합니다."
    )


if len(valid_df) != EXPECTED_VALID:

    raise RuntimeError(

        "\nValidation 데이터 연결 개수가 맞지 않습니다.\n"

        f"예상: {EXPECTED_VALID:,}\n"

        f"현재: {len(valid_df):,}\n"

        "학습을 시작하지 않고 중단합니다."
    )


print(
    "\n데이터 연결 완료"
)

print(
    "Training:",
    f"{len(train_df):,}"
)

print(
    "Validation:",
    f"{len(valid_df):,}"
)



# =============================================================================
# 2. Dataset과 DataLoader 구성
# =============================================================================

display(Markdown("""
# 2. Dataset과 DataLoader 구성

PyTorch가 기존 이미지를 학습에 사용할 수 있도록  
`Dataset`과 `DataLoader`를 구성한다.

현재 이미지는 grayscale 1채널이지만,  
MobileNetV3 Small, EfficientNetB0, ResNet18은 기본적으로 3채널 입력을 사용한다.

따라서 같은 grayscale 값을 3개 채널에 복제한다.  
**컬러 이미지로 되돌리는 것은 아니다.**

`DataLoader`는 전체 데이터를 한꺼번에 GPU에 올리지 않고  
**32장씩 Batch 단위로 모델에 전달**한다.
"""))



# -----------------------------------------------------------------------------
# 이미지 변환
# -----------------------------------------------------------------------------

transform = transforms.Compose([

    # grayscale 값을 동일하게 3개 채널로 복제
    transforms.Grayscale(
        num_output_channels=3
    ),

    # 0~255 픽셀값을 0~1 범위 Tensor로 변환
    transforms.ToTensor()
])



# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------

class EmotionDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = (
            transform
        )


    def __len__(
        self
    ):

        # 전체 이미지 개수
        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = (
            self.dataframe
            .iloc[index]
        )


        # 이미 전처리된 grayscale PNG 읽기
        image = (
            Image
            .open(
                row["image_path"]
            )
            .convert("L")
        )


        # 문자열 감정이 아니라
        # 숫자로 변환된 정답 사용
        label = int(
            row["label_index"]
        )


        if self.transform is not None:

            image = (
                self.transform(
                    image
                )
            )


        return image, label



# -----------------------------------------------------------------------------
# Training / Validation Dataset
# -----------------------------------------------------------------------------

train_dataset = EmotionDataset(

    train_df,

    transform=transform
)


valid_dataset = EmotionDataset(

    valid_df,

    transform=transform
)



# -----------------------------------------------------------------------------
# DataLoader
#
# batch_size=32
# → 이미지 32장씩 모델에 전달
#
# shuffle=True
# → Training 데이터 순서를 매 Epoch 섞음
#
# num_workers=0
# → Windows + Jupyter 안정성을 위해 메인 프로세스에서 로딩
# -----------------------------------------------------------------------------

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=True
)


valid_loader = DataLoader(

    valid_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True
)



print(
    "\nTraining Dataset:",
    f"{len(train_dataset):,}"
)


print(
    "Validation Dataset:",
    f"{len(valid_dataset):,}"
)


print(
    "Batch Size:",
    BATCH_SIZE
)



# =============================================================================
# 3. 모델 구성
# =============================================================================

display(Markdown("""
# 3. 모델 구성

팀에서 선택한 다음 3개 모델을 동일한 조건으로 비교한다.

- **MobileNetV3 Small**
- **EfficientNetB0**
- **ResNet18**

이번 본학습에서는 `weights=None`을 사용한다.

즉,

- ImageNet 사전학습 가중치를 사용하지 않는다.
- 이전 1 Epoch 스크리닝 모델도 불러오지 않는다.
- 각 모델은 새로운 가중치 상태에서 처음부터 학습한다.
"""))



def create_model(
    model_name
):


    # -------------------------------------------------------------------------
    # MobileNetV3 Small
    # -------------------------------------------------------------------------

    if model_name == (
        "mobilenet_v3_small"
    ):

        model = (
            models
            .mobilenet_v3_small(
                weights=None
            )
        )


        # 마지막 출력층을
        # 1000개 ImageNet 클래스가 아니라
        # 우리 프로젝트의 7개 감정으로 변경
        model.classifier[3] = (
            nn.Linear(

                model
                .classifier[3]
                .in_features,

                NUM_CLASSES
            )
        )



    # -------------------------------------------------------------------------
    # EfficientNetB0
    # -------------------------------------------------------------------------

    elif model_name == (
        "efficientnet_b0"
    ):

        model = (
            models
            .efficientnet_b0(
                weights=None
            )
        )


        model.classifier[1] = (
            nn.Linear(

                model
                .classifier[1]
                .in_features,

                NUM_CLASSES
            )
        )



    # -------------------------------------------------------------------------
    # ResNet18
    # -------------------------------------------------------------------------

    elif model_name == (
        "resnet18"
    ):

        model = (
            models
            .resnet18(
                weights=None
            )
        )


        model.fc = (
            nn.Linear(

                model.fc.in_features,

                NUM_CLASSES
            )
        )



    else:

        raise ValueError(
            "지원하지 않는 모델입니다."
        )


    # 생성한 모델을 GPU로 이동
    model = model.to(
        device
    )


    return model



# =============================================================================
# 4. 모델 본학습
# =============================================================================

display(Markdown("""
# 4. 모델 본학습

세 모델을 전체 Training 데이터로 순차 학습한다.

### 공통 학습 조건

- **Optimizer:** Adam
- **Learning Rate:** 0.001
- **Batch Size:** 32
- **Epoch:** 최대 30
- **EarlyStopping:** patience 5
- **Pretrained:** 사용하지 않음
- **Scheduler:** 사용하지 않음
- **Weight Decay:** 사용하지 않음

각 Epoch에서 Training과 Validation의 Loss와 Accuracy를 계산한다.

Validation Loss가 가장 낮아진 모델을 Best Model로 저장하고,  
5 Epoch 동안 Validation Loss가 개선되지 않으면 EarlyStopping으로 종료한다.
"""))



def train_model(
    model_name
):


    print(
        "\n"
        + "=" * 70
    )

    print(
        f"{model_name} 본학습 시작"
    )

    print(
        "=" * 70
    )



    # -------------------------------------------------------------------------
    # 새로운 모델 생성
    # -------------------------------------------------------------------------

    model = create_model(
        model_name
    )



    # -------------------------------------------------------------------------
    # 손실 함수
    #
    # CrossEntropyLoss:
    # 7개 감정 중 하나를 분류하는
    # 다중 클래스 분류에 사용
    # -------------------------------------------------------------------------

    criterion = (
        nn.CrossEntropyLoss()
    )



    # -------------------------------------------------------------------------
    # Optimizer
    #
    # Adam
    # Learning Rate = 0.001
    # -------------------------------------------------------------------------

    optimizer = (
        torch.optim.Adam(

            model.parameters(),

            lr=LEARNING_RATE
        )
    )



    # 가장 좋은 Validation Loss
    best_val_loss = float(
        "inf"
    )


    best_val_acc = 0.0

    best_epoch = 0


    # EarlyStopping 횟수
    early_stop_count = 0


    # Epoch별 기록
    history = []



    # -------------------------------------------------------------------------
    # 저장 경로
    # -------------------------------------------------------------------------

    model_path = (

        MODEL_DIR
        /
        f"{model_name}_best.pth"
    )


    history_path = (

        OUTPUT_DIR
        /
        f"{model_name}_history.csv"
    )



    # =========================================================================
    # Epoch 반복
    # =========================================================================

    for epoch in range(
        1,
        EPOCHS + 1
    ):


        start_time = (
            time.time()
        )



        # =====================================================================
        # Training
        # =====================================================================

        model.train()


        train_loss_sum = 0.0

        train_correct = 0

        train_total = 0



        for images, labels in train_loader:


            # -------------------------------------------------------------
            # Batch 데이터를 GPU로 이동
            # -------------------------------------------------------------

            images = images.to(

                device,

                non_blocking=True
            )


            labels = labels.to(

                device,

                non_blocking=True
            )



            # -------------------------------------------------------------
            # 이전 Batch의 Gradient 초기화
            # -------------------------------------------------------------

            optimizer.zero_grad()



            # -------------------------------------------------------------
            # 순전파
            #
            # 이미지를 모델에 입력하여
            # 7개 감정에 대한 예측값 생성
            # -------------------------------------------------------------

            outputs = model(
                images
            )



            # -------------------------------------------------------------
            # 예측값과 실제 정답의 차이 계산
            # -------------------------------------------------------------

            loss = criterion(

                outputs,

                labels
            )



            # -------------------------------------------------------------
            # 역전파
            # -------------------------------------------------------------

            loss.backward()



            # -------------------------------------------------------------
            # Adam으로 가중치 업데이트
            # -------------------------------------------------------------

            optimizer.step()



            # -------------------------------------------------------------
            # Loss 누적
            # -------------------------------------------------------------

            train_loss_sum += (

                loss.item()
                *
                images.size(0)
            )



            # -------------------------------------------------------------
            # 가장 높은 값을 가진 감정을 최종 예측으로 선택
            # -------------------------------------------------------------

            predictions = (
                outputs.argmax(
                    dim=1
                )
            )



            train_correct += (

                predictions
                .eq(labels)
                .sum()
                .item()
            )


            train_total += (
                labels.size(0)
            )



        # ---------------------------------------------------------------------
        # Training 전체 평균 Loss / Accuracy
        # ---------------------------------------------------------------------

        train_loss = (

            train_loss_sum
            /
            train_total
        )


        train_acc = (

            train_correct
            /
            train_total
        )



        # =====================================================================
        # Validation
        # =====================================================================

        model.eval()


        val_loss_sum = 0.0

        val_correct = 0

        val_total = 0



        # Validation에서는 가중치 업데이트를 하지 않는다.
        with torch.no_grad():


            for images, labels in valid_loader:


                images = images.to(

                    device,

                    non_blocking=True
                )


                labels = labels.to(

                    device,

                    non_blocking=True
                )



                outputs = model(
                    images
                )



                loss = criterion(

                    outputs,

                    labels
                )



                val_loss_sum += (

                    loss.item()
                    *
                    images.size(0)
                )



                predictions = (
                    outputs.argmax(
                        dim=1
                    )
                )



                val_correct += (

                    predictions
                    .eq(labels)
                    .sum()
                    .item()
                )


                val_total += (
                    labels.size(0)
                )



        # ---------------------------------------------------------------------
        # Validation 전체 평균 Loss / Accuracy
        # ---------------------------------------------------------------------

        val_loss = (

            val_loss_sum
            /
            val_total
        )


        val_acc = (

            val_correct
            /
            val_total
        )



        # Epoch 소요 시간
        elapsed = (

            time.time()
            -
            start_time
        )



        # =====================================================================
        # Epoch 결과 기록
        # =====================================================================

        history.append({

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_acc":
                train_acc,

            "val_loss":
                val_loss,

            "val_acc":
                val_acc,

            "minutes":
                elapsed / 60
        })



        # 매 Epoch 결과를 저장
        # 중간에 프로그램이 종료되어도 완료된 기록은 남는다.
        pd.DataFrame(
            history
        ).to_csv(

            history_path,

            index=False,

            encoding="utf-8-sig"
        )



        # ---------------------------------------------------------------------
        # 현재 Epoch 결과 출력
        # ---------------------------------------------------------------------

        print(

            f"Epoch "
            f"{epoch:02d}/"
            f"{EPOCHS} | "

            f"train_loss="
            f"{train_loss:.4f} | "

            f"train_acc="
            f"{train_acc:.4f} | "

            f"val_loss="
            f"{val_loss:.4f} | "

            f"val_acc="
            f"{val_acc:.4f} | "

            f"time="
            f"{elapsed / 60:.1f}분"
        )



        # =====================================================================
        # Best Model 저장
        # =====================================================================

        if val_loss < best_val_loss:


            # Validation Loss가 좋아졌으므로
            # Best 값 갱신
            best_val_loss = (
                val_loss
            )


            best_val_acc = (
                val_acc
            )


            best_epoch = (
                epoch
            )


            # EarlyStopping 카운트 초기화
            early_stop_count = 0



            # 모델 가중치 저장
            torch.save(

                {

                    "epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "val_loss":
                        val_loss,

                    "val_acc":
                        val_acc

                },

                model_path
            )



            print(
                "→ Best Model 저장"
            )



        else:


            # Validation Loss가 개선되지 않은 횟수 증가
            early_stop_count += 1


            print(

                "→ Validation Loss 개선 없음 "

                f"("
                f"{early_stop_count}"
                f"/"
                f"{PATIENCE}"
                f")"
            )



        # =====================================================================
        # EarlyStopping
        # =====================================================================

        if early_stop_count >= PATIENCE:


            print(
                "→ EarlyStopping 실행"
            )


            break



    # =========================================================================
    # 한 모델 학습 완료
    # =========================================================================

    print(
        f"\n{model_name} 학습 완료"
    )


    print(
        "Best Epoch:",
        best_epoch
    )


    print(
        "Best Val Loss:",
        round(
            best_val_loss,
            4
        )
    )


    print(
        "Best Val Acc:",
        round(
            best_val_acc,
            4
        )
    )



    result = {

        "model":
            model_name,

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,

        "best_val_acc":
            best_val_acc
    }



    # -------------------------------------------------------------------------
    # 다음 모델 학습을 위한 메모리 정리
    # -------------------------------------------------------------------------

    del model

    del optimizer


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()



    return result



# =============================================================================
# 세 모델 순차 본학습
# =============================================================================

MODEL_NAMES = [

    "mobilenet_v3_small",

    "efficientnet_b0",

    "resnet18"
]



results = []



for model_name in MODEL_NAMES:


    result = train_model(
        model_name
    )


    results.append(
        result
    )



# =============================================================================
# 5. 최종 학습 결과 저장
# =============================================================================

display(Markdown("""
# 5. 학습 결과 저장

세 모델의 Best 결과를 하나의 표로 정리한다.

저장되는 파일은 다음과 같다.

- `05_models` : 각 모델의 Best `.pth`
- `06_outputs` : 각 Epoch 학습 기록 `.csv`
- `06_outputs/pytorch_training_summary.csv` : 세 모델 최종 비교표

기존 `02_data`의 이미지와 라벨 파일은 수정하지 않는다.
"""))



# 결과 DataFrame
results_df = pd.DataFrame(
    results
)



# Validation Loss가 낮은 순서로 정렬
results_df = (

    results_df

    .sort_values(
        "best_val_loss"
    )

    .reset_index(
        drop=True
    )
)



summary_path = (

    OUTPUT_DIR
    /
    "pytorch_training_summary.csv"
)



results_df.to_csv(

    summary_path,

    index=False,

    encoding="utf-8-sig"
)



print(
    "\n"
    + "=" * 70
)


print(
    "세 모델 본학습 완료"
)


print(
    "=" * 70
)



display(
    results_df
)



print(
    "\n최종 결과 저장:",
    summary_path
)


# 1. 학습 환경 및 기존 데이터 연결

이미 전처리가 완료된 **224×224 grayscale PNG**와  
기존 AI-Hub JSON의 감정 라벨을 연결한다.

- 이미지 resize를 다시 하지 않는다.
- grayscale 변환을 다시 하지 않는다.
- 기존 이미지 파일을 복사하거나 이동하지 않는다.
- `filename`을 기준으로 이미지와 라벨을 연결한다.
- 감정 정답은 기존 AI-Hub의 `faceExp_uploader`를 사용한다.
- Training 전체 223,578장과 Validation 전체 52,126장을 사용한다.


사용 장치: cuda
GPU: NVIDIA GeForce RTX 4060

Training 라벨 읽는 중...
Training 라벨: 417,167

Validation 라벨 읽는 중...
Validation 라벨: 52,126

Training grayscale PNG와 라벨 연결 중...
Training 연결: 223,578

Validation grayscale PNG와 라벨 연결 중...
Validation 연결: 52,126

데이터 연결 완료
Training: 223,578
Validation: 52,126



# 2. Dataset과 DataLoader 구성

PyTorch가 기존 이미지를 학습에 사용할 수 있도록  
`Dataset`과 `DataLoader`를 구성한다.

현재 이미지는 grayscale 1채널이지만,  
MobileNetV3 Small, EfficientNetB0, ResNet18은 기본적으로 3채널 입력을 사용한다.

따라서 같은 grayscale 값을 3개 채널에 복제한다.  
**컬러 이미지로 되돌리는 것은 아니다.**

`DataLoader`는 전체 데이터를 한꺼번에 GPU에 올리지 않고  
**32장씩 Batch 단위로 모델에 전달**한다.



Training Dataset: 223,578
Validation Dataset: 52,126
Batch Size: 32



# 3. 모델 구성

팀에서 선택한 다음 3개 모델을 동일한 조건으로 비교한다.

- **MobileNetV3 Small**
- **EfficientNetB0**
- **ResNet18**

이번 본학습에서는 `weights=None`을 사용한다.

즉,

- ImageNet 사전학습 가중치를 사용하지 않는다.
- 이전 1 Epoch 스크리닝 모델도 불러오지 않는다.
- 각 모델은 새로운 가중치 상태에서 처음부터 학습한다.


...


mobilenet_v3_small 본학습 시작
Epoch 01/30 | train_loss=1.9476 | train_acc=0.1423 | val_loss=1.9625 | val_acc=0.1429 | time=56.2분
→ Best Model 저장
Epoch 02/30 | train_loss=1.9473 | train_acc=0.1440 | val_loss=1.9468 | val_acc=0.1441 | time=10.1분
→ Best Model 저장
Epoch 03/30 | train_loss=1.9471 | train_acc=0.1417 | val_loss=1.9513 | val_acc=0.1439 | time=10.1분
→ Validation Loss 개선 없음 (1/5)
Epoch 04/30 | train_loss=1.9469 | train_acc=0.1426 | val_loss=1.9465 | val_acc=0.1438 | time=10.1분
→ Best Model 저장
Epoch 05/30 | train_loss=1.9469 | train_acc=0.1436 | val_loss=1.9468 | val_acc=0.1422 | time=10.1분
→ Validation Loss 개선 없음 (1/5)
Epoch 06/30 | train_loss=1.9469 | train_acc=0.1417 | val_loss=1.9464 | val_acc=0.1439 | time=10.1분
→ Best Model 저장
Epoch 07/30 | train_loss=1.9469 | train_acc=0.1435 | val_loss=1.9475 | val_acc=0.1430 | time=10.1분
→ Validation Loss 개선 없음 (1/5)
Epoch 08/30 | train_loss=1.9468 | train_acc=0.1427 | val_loss=1.9465 | val_acc=0.1427 | time=10.1분
→ Validation Loss 개선 없음 (2/


# 5. 학습 결과 저장

세 모델의 Best 결과를 하나의 표로 정리한다.

저장되는 파일은 다음과 같다.

- `05_models` : 각 모델의 Best `.pth`
- `06_outputs` : 각 Epoch 학습 기록 `.csv`
- `06_outputs/pytorch_training_summary.csv` : 세 모델 최종 비교표

기존 `02_data`의 이미지와 라벨 파일은 수정하지 않는다.



세 모델 본학습 완료


,model,best_epoch,best_val_loss,best_val_acc
0,resnet18,4,1.945983,0.142501
1,efficientnet_b0,6,1.946025,0.143000
2,mobilenet_v3_small,11,1.946065,0.143805



최종 결과 저장: D:\emotion_recognition_project\06_outputs\pytorch_training_summary.csv
